Open and read data

In [3]:
%pip install pandas
%pip install plotly

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import plotly.express as px
import seaborn as sns

import dash
from dash import dcc, html

import logging

In [5]:
logging.basicConfig(
    level=logging.INFO,  # Set the minimum log level
    format='%(asctime)s - %(levelname)s - %(message)s',  # Log message format
    # filename='app.log'  # Log to a file (optional)
)

In [6]:
class LoadDataset:
    """ 
    passing the raw dataset file path as a dependency
    """
    def __init__(self, raw_file_path):
        self.raw_file_path = raw_file_path

    def read_file(self):
        """
        handling file existence, empty dataset, and parsing errors
        """
        try:
            return pd.read_csv(self.raw_file_path, encoding="ISO-8859-1", engine='python')
            logging.info("the dataset is opened successfully")
        except FileNotFoundError:
            logging.error("Error: The file was not found.")
        except pd.errors.EmptyDataError:
            logging.error("Error: The file is empty.")
        except pd.errors.ParserError:
            logging.error("Error: The file could not be parsed.")
        

In [7]:
class DataCleaner:
    
    def __init__(self, data):
        self.data = data


    """
    since 15% of rows contain null values, 
    they will be replaced with the mean (float and int value) and the mode (string values) instead of being dropped
    """
    
    def imputate_null_values(self):
        # Impute missing values with the mean for numerical columns
        try:
            for column in self.data.select_dtypes(include=['float64', 'int64']).columns:
                self.data[column].fillna(self.data[column].mean(), inplace=True)

        except Exception as e:
            logging.error("Failed to replace imputate some numeric values {e}")
        
        # Impute missing values with the mode for string (categorical) columns
        try:
            for column in self.data.select_dtypes(include=['object']).columns:
                self.data[column].fillna(self.data[column].mode()[0], inplace=True)
        except Exception as e:
            logging.error("Failed to imputate some alphabetic values {e}")
            
        return self.data


    def remove_duplicates(self):
        # Drop rows with missing values in 'Brand', 'Model', and 'Trim'
        self.data = self.data.dropna(subset=['make', 'model', 'trim'])
        
        # Calculate duplicate mask based on specific columns
        duplicate_mask = self.data.duplicated(subset=['make', 'model', 'trim'], keep='first')
        duplicate_rows_num = duplicate_mask.sum()

        # Print debug information
        print(f"Number of rows before removing duplicates: {len(self.data)}")
        print(f"Number of duplicate rows found: {duplicate_rows_num}")
        
        # Remove duplicate rows
        self.data = self.data.drop_duplicates(subset=['make', 'model', 'trim'], keep='first')
        
        # Print final number of rows
        print(f"Number of rows after removing duplicates: {len(self.data)}")
        
        return self.data
        

    
    def rename_headers(self):
        try:
            headers_replacements = {
            "make":"Brand",
            "sellingprice":"SellingPrice",
            "saledate":"SaleDate",
            }
            self.data.rename(columns=headers_replacements, inplace=True)
            
            self.data.columns = self.data.columns.str.capitalize()

            logging.info("renamed headers successfully")
            return self.data

        except Exception as e:
            logging.error(f"Failed to rename some header titles: {e}")

    
    def capitalise_headers(self):
        """
        - capitalising dataframe's headers for convenience
        """
        try:
            self.data.columns = self.data.columns.str.title()
            logging.info("capitalised headers successfully")
            return self.data

        except Exception as e:
            logging.error(f"Failed to capitalise header titles: {e}")


    def validate_values(self):
        """ 
        - Ensuring dataframe's attributes contain the proper data types
        """
        data_types = {
            "Year": "int64",
            "Brand": "object",
            "Model": "object",
            "Trim": "object",
            "Transmission":"object",
            "Vin": "object",
            "State":"object",
            "Condition":"float64",
            "Odometer":"float64",
            "Color":"object",
            "Interior":"object",
            "Seller":"object",
            "Mmr":"float64",
            "Sellingprice": "float64",
            "Saledate": "object"
        }
        
        try:
            for column, d_type in data_types.items():
                if self.data[column].dtype != d_type:
                    logging.info(f"Validation failed for column '{column}'. Expected {d_type}, got {self.data[column].dtype}")
                else:
                    logging.info(f"Validation passed for column '{column}'. Expected {d_type}, got {self.data[column].dtype}")
        except Exception as e:
            logging.error(f"failed to convert to data type {e}")
                


    def normalise_brands(self):
        brand_mapping = {
            'vw': 'Volkswagen',
            'volkswagen': 'Volkswagen',
            'lincoln': 'Lincoln',
            'Lincoln': 'Lincoln',
            'mercedes-b': 'Mercedes-Benz',
            'ford': 'Ford',
            'ford tk': 'Ford',
            'dodge': 'Dodge',
            'dodge tk': 'Dodge',
        }
    
        try:
            # Convert the 'Brand' column to lowercase
            self.data['Brand'] = self.data['Brand'].str.lower()
        
            # Apply the mapping to the 'Brand' column
            self.data['Brand'] = self.data['Brand'].replace(brand_mapping)

            logging.info("normalised brands susccesfully")

        except Exception as e:
            logging.error(f"failed to normalise brand, {e}")
        
        return self.data


    
    def normalise_states(self):
        us_states = {
            "ak": "Alaska", "al": "Alabama", "ar": "Arkansas", "az": "Arizona",
            "ca": "California", "co": "Colorado", "ct": "Connecticut", "de": "Delaware",
            "fl": "Florida", "ga": "Georgia", "hi": "Hawaii", "ia": "Iowa",
            "id": "Idaho", "il": "Illinois", "in": "Indiana", "ks": "Kansas",
            "ky": "Kentucky", "la": "Louisiana", "ma": "Massachusetts", "md": "Maryland",
            "me": "Maine", "mi": "Michigan", "mn": "Minnesota", "mo": "Missouri",
            "ms": "Mississippi", "mt": "Montana", "nc": "North Carolina", "nd": "North Dakota",
            "ne": "Nebraska", "nh": "New Hampshire", "nj": "New Jersey", "nm": "New Mexico",
            "nv": "Nevada", "ny": "New York", "oh": "Ohio", "ok": "Oklahoma",
            "or": "Oregon", "pa": "Pennsylvania", "ri": "Rhode Island", "sc": "South Carolina",
            "sd": "South Dakota", "tn": "Tennessee", "tx": "Texas", "ut": "Utah",
            "va": "Virginia", "vt": "Vermont", "wa": "Washington", "wi": "Wisconsin",
            "wv": "West Virginia", "wy": "Wyoming", "dc": "District of Columbia",
            "as": "American Samoa", "gu": "Guam", "mp": "Northern Mariana Islands",
            "pr": "Puerto Rico", "vi": "U.S. Virgin Islands"
        }
        
        try:
            self.data['State'] = self.data['State'].str.lower()
            self.data['State'] = self.data['State'].apply(lambda x: us_states.get(x, 'unknown'))
            logging.info("normalised states succesffulyy")
        except Exception as e:
            logging.error(f"failed to normalised states, {e}")
        
        return self.data



    def normalise_transmissions(self):
        normal_transmissions = ['automatic', 'manual']
        try:
            self.data['Transmission'] = self.data['Transmission'].str.lower()
            self.data['Transmission'] = self.data['Transmission'].apply(lambda x: x if x in normal_transmissions else 'other')
            logging.info("Normalised transmissions succesfully")
        except Exception as e:
            loggin.error(f"failed to normalise transmission, {e}")
        

    def normalise_colors(self):
        """
        - Since some oddly-named values were found during data exploration, I decided to 
        keep the normal names and replace everything else with "unknown"
        
        - Some of the integer values used as "Color" show no sign of being either hex or rgb values so 
        converting them to such values was not a consideration
        
        """
        normal_colors = [
            'red', 'white', 'gray', 'black', 'burgundy', 'silver', 'brown', 'blue', 'beige',
            'purple', 'gold', 'yellow', 'green', 'charcoal', 'orange', 'off white', 'turquoise', 
            'pink', 'lime'
        ]

        try:
            self.data['Color'] = self.data['Color'].str.lower()
            self.data['Color'] = self.data['Color'].apply(lambda x: x if x in normal_colors else 'unknown')
            logging.info("Normalised colors successfully")
        except Exception as e:
            logging.error(f"failed to normalise some colors, {e}")
                
                
    def save_changes(self):
        try:
            self.data.to_csv('../dataset/cleaned/cleaned_car_sales_data.csv', index=False)
            logging.info("saved the dataset successfully")
        except Exception as e:
            logging.error(f"failed to save the new dataset file, {e}")
            
        return self.data
        

In [8]:
class DataExplorer:
    def __init__(self, cleaned_data):
        self.cleaned_data = cleaned_data


    def format_revenue(self, number):
        if isinstance(number, str):
            number = float(number.replace(',', ''))
        elif not isinstance(number, (int, float)):
            raise ValueError("Input must be a string, int, or float.")
        
        if number >= 1e9:
            return f"{number/1e9:.2f}B"
        elif number >= 1e6:
            return f"{number/1e6:.2f}M"
        else:
            return f"{number:.2f}"


    def plot_figure(self, data, plot_type='line', x_col=None, y_col=None, title='', labels=None, custom_hover=False, width=1000, height=600):
        if plot_type == 'line':
            try:
                fig = px.line(data, x=x_col, y=y_col, title=title, markers=True, labels=labels)
                logging.info("created line chart succesfully")
            except Exception as e:
                logging.error(f"failed to create line chart, {e}")
        elif plot_type == 'bar':
            try:
                fig = px.bar(data, x=x_col, y=y_col, title=title, labels=labels)
                logging.info("created bar chart succesfully")
            except Exception as e:
                logging.error(f"failed to create bar chart, {e}")
            
        
        if custom_hover:
            fig.update_traces(hovertemplate=f'{x_col}: %{{x}}<br>{y_col}: %{{y:$,.2f}}')

        fig.update_layout(xaxis_title=x_col, yaxis_title=y_col, width=width, height=height)
        fig.show()
        return fig


        

    
    def sales_by_year(self):
        annual_sales = (
            self.cleaned_data.groupby('Year')['Sellingprice']
            .sum()
            .reset_index()
            .rename(columns={'Sellingprice': 'Revenue'})
        )
        annual_sales = annual_sales.sort_values(by='Year', ascending=True)

        # Change index to start from 1
        annual_sales.index = annual_sales.index + 1
 

        self.plot_figure(
            annual_sales,
            plot_type='line',
            x_col='Year',
            y_col='Revenue',
            title='Revenue over Time'
        )
        
        return annual_sales

    
    def top_10_selling_brands(self):
        top_10_selling_brands = (
            self.cleaned_data.groupby('Brand')['Sellingprice']
            .sum()
            .reset_index()
            .rename(columns={'Sellingprice': 'Revenue'})
        )
        
        top_10_selling_brands = top_10_selling_brands.sort_values(by='Revenue', ascending=False)
    
        top_10_selling_brands.index = top_10_selling_brands.index + 1
    
        self.plot_figure(
            top_10_selling_brands.head(10),
            plot_type='bar',
            x_col='Brand',
            y_col='Revenue',
            title='Top 10 Selling Car Brands'
        )
    
        return top_10_selling_brands.head(10)

    
    def top_10_most_expensive_cars(self):
        # Concatenate 'Brand', 'Model', and 'Trim' into a new column 'CarName'
        self.cleaned_data['CarName'] = self.cleaned_data.apply(lambda row: f"{row['Brand']} {row['Model']} {row['Trim']} {row['Year']}", axis=1)
    
        # Group by 'CarName' and calculate the sum of 'Sellingprice'
        top_10_most_expensive_cars = (
            self.cleaned_data.groupby('CarName')['Sellingprice']
            .sum()
            .reset_index()
            .rename(columns={'Sellingprice': 'Price'})
        )
    
        # Sort by 'Revenue' in descending order
        top_10_most_expensive_cars = top_10_most_expensive_cars.sort_values(by='Price', ascending=False)
    
        # Apply formatting to the Revenue column
        top_10_most_expensive_cars['FormattedPrice'] = top_10_most_expensive_cars['Price'].apply(self.format_revenue)
    
        # Plotting top 10 selling cars by revenue
        fig = px.bar(top_10_most_expensive_cars.head(10), x='CarName', y='Price', title='Top 10 Most Expensive Cars')
        fig.update_layout(width=1000, height=1000)
        fig.show()
    
        # Return the top 10 cars with revenue information
        return top_10_most_expensive_cars.head(10)

   

    def sales_by_transmission(self):
        try:
            sales_by_transmission = (
                self.cleaned_data.groupby('Transmission')['Sellingprice']
                .sum()
                .reset_index()
                .rename(columns={'Sellingprice': 'Revenue'})
            )
            
            sales_by_transmission = sales_by_transmission.sort_values(by='Revenue', ascending=False)
            sales_by_transmission.index = sales_by_transmission.index + 1
            sales_by_transmission['FormattedRevenue'] = sales_by_transmission['Revenue'].apply(self.format_revenue)
            
            # Plotting sales by transmission type as a pie chart
            fig_pie = px.pie(sales_by_transmission, names='Transmission', values='Revenue', title='Sales by Transmission Type')
            fig_pie.update_layout(width=800, height=800)
            fig_pie.show()
    
            return sales_by_transmission
    
        except Exception as e:
            logging.error(f"An error occurred: {e}")
            return None

     
    
    def total_sales(self):
        total_sales = self.cleaned_data['Sellingprice'].sum()
        # formatting the total sales
        formatted_total_sales = self.format_revenue(total_sales)
        
        return formatted_total_sales

    
    

    def top_10_sellers_by_revenue(self):
        top_10_sellers_by_revenue = (
            self.cleaned_data.groupby('Seller')['Sellingprice']
            .sum()
            .reset_index()
            .rename(columns={'Sellingprice': 'Revenue'})
        )
        top_10_sellers_by_revenue = top_10_sellers_by_revenue.sort_values(by='Revenue', ascending=False)
        top_10_sellers_by_revenue.index = top_10_sellers_by_revenue.index + 1

        top_10_sellers_by_revenue['FormattedRevenue'] = top_10_sellers_by_revenue['Revenue'].apply(self.format_revenue)

        # Plotting top 10 selling cars by revenue
        fig = px.bar(top_10_sellers_by_revenue.head(10), x='Seller', y='Revenue', title='Top 10 Sellers')
        fig.update_layout(width=800, height=800)
        fig.show()

        return top_10_sellers_by_revenue.head(10)


    
    def top_10_worst_sellers_by_revenue(self):
        top_10_worst_sellers_by_revenue = (
            self.cleaned_data.groupby('Seller')['Sellingprice']
            .sum()
            .reset_index()
            .rename(columns={'Sellingprice': 'Revenue'})
        )
        top_10_worst_sellers_by_revenue = top_10_worst_sellers_by_revenue.sort_values(by='Revenue', ascending=False)
        top_10_worst_sellers_by_revenue.index = top_10_worst_sellers_by_revenue.index + 1

        top_10_worst_sellers_by_revenue['FormattedRevenue'] = top_10_worst_sellers_by_revenue['Revenue'].apply(self.format_revenue)

        # Plotting top 10 selling cars by revenue
        fig = px.bar(top_10_worst_sellers_by_revenue.tail(10), x='Seller', y='Revenue', title='Top 10 Worst Sellers')
        fig.update_layout(width=800, height=800)
        fig.show()

        return top_10_worst_sellers_by_revenue.tail(10)




    def top_10_most_profitable_cars(self):
        # Creating an identifier for unique vehicles
        self.cleaned_data['CarName'] = self.cleaned_data.apply(lambda row: f"{row['Brand']} {row['Model']} {row['Trim']} {row['Year']} {row['Transmission']}", axis=1)
        
        
        top_10_selling_cars = (
            self.cleaned_data.groupby(['Vin', 'CarName'])['Sellingprice']
            .sum()
            .reset_index()
            .rename(columns={'Sellingprice': 'Revenue'})
        )
        
        top_10_selling_cars = top_10_selling_cars.sort_values(by='Revenue', ascending=False)
        top_10_selling_cars.index = top_10_selling_cars.index + 1

        top_10_selling_cars['FormattedRevenue'] = top_10_selling_cars['Revenue'].apply(self.format_revenue)

        # Plotting top 10 selling cars by revenue
        fig = px.bar(top_10_selling_cars.head(10), x='CarName', y='FormattedRevenue', title='Top 10 Most Profitable Selling Cars')
        fig.update_layout(width=800, height=800)
        fig.show()

        return top_10_selling_cars.head(10)

    
    def top_10_least_profitable_cars(self):
        # Creating an identifier for unique vehicles
        self.cleaned_data['CarName'] = self.cleaned_data.apply(lambda row: f"{row['Brand']} {row['Model']} {row['Trim']} {row['Year']} {row['Transmission']}", axis=1)
 
        least_profitable_cars = (
            self.cleaned_data.groupby(['Vin', 'CarName'])['Sellingprice']
            .sum()
            .reset_index()
            .rename(columns={'Sellingprice': 'Revenue'})
        )
        
        least_profitable_cars = least_profitable_cars.sort_values(by='Revenue', ascending=False)
        least_profitable_cars.index = least_profitable_cars.index + 1

        least_profitable_cars['FormattedRevenue'] = least_profitable_cars['Revenue'].apply(self.format_revenue)

    

        return least_profitable_cars.tail(10)
        


    
    def revenue_by_state(self):
        revenue_by_state = (
            self.cleaned_data.groupby('State')['Sellingprice']
            .sum()
            .reset_index()
            .rename(columns={'Sellingprice': 'Revenue'})
        )
        
        revenue_by_state = revenue_by_state.sort_values(by='Revenue', ascending=False)
        revenue_by_state.index = revenue_by_state.index + 1
    
        revenue_by_state['FormattedRevenue'] = revenue_by_state['Revenue'].apply(self.format_revenue)
    
        # Plotting revenue by state
        fig = px.bar(revenue_by_state.head(10), x='State', y='Revenue', title='Revenue by State')
        fig.update_layout(width=800, height=800)
        fig.show()
    
        return revenue_by_state.head(10)




    def average_prices_by_state(self):
        # Ensure State column is consistent with full state names
        self.cleaned_data['State'] = self.cleaned_data['State'].str.title()
    
        average_prices_by_state = (
            self.cleaned_data.groupby('State')['Sellingprice']
            .mean()
            .reset_index()
            .rename(columns={'Sellingprice': 'AveragePrices'})
        )
        
        average_prices_by_state = average_prices_by_state.sort_values(by='AveragePrices', ascending=False)
     
        average_prices_by_state['FormattedPrices'] = average_prices_by_state['AveragePrices'].apply(self.format_revenue)
        
        # Plotting average prices by state
        fig = px.bar(average_prices_by_state.head(10), x='State', y='AveragePrices', title='Average Prices by State')
        fig.update_layout(width=800, height=800)
        fig.show()
        
        return average_prices_by_state.head(10)


    def most_popular_colors(self):
        popular_colors = (
            self.cleaned_data['Color']
            .value_counts()
        )

        return popular_colors.head(10)

    

SyntaxError: positional argument follows keyword argument (4051723383.py, line 89)

In [ ]:
import pandas as pd
import plotly.express as px

class DataCorrelation:
    def __init__(self, cleaned_data):
        self.cleaned_data = cleaned_data
 

    def revenue_price_correlation(self):
        """
        Calculates and visualizes the correlation between average selling price and revenue by state.
        """
        try:
            # Calculate revenue by state
            revenue = (
                self.cleaned_data.groupby('State')['Sellingprice']
                .sum()
                .reset_index()
                .rename(columns={'Sellingprice': 'Revenue'})
            )
            
            # Calculate average selling price by state
            avg_price = (
                self.cleaned_data.groupby('State')['Sellingprice']
                .mean()
                .reset_index()
                .rename(columns={'Sellingprice': 'AveragePrice'})
            )
    
            # Merge revenue and average selling price
            merged_df = pd.merge(revenue, avg_price, on='State', how='left')
    
            # Calculate the correlation between 'AveragePrice' and 'Revenue'
            correlation = merged_df['Revenue'].corr(merged_df['AveragePrice'])
    
            # Visualization
            fig = px.scatter(
                merged_df, 
                x='AveragePrice', 
                y='Revenue', 
                trendline='ols', 
                title=f'Correlation between Average Selling Price and Revenue: {correlation:.2f}',
                labels={'AveragePrice': 'Average Selling Price', 'Revenue': 'Revenue'}
            )
            fig.update_layout(width=800, height=600)
            fig.show()
    
            print(f"Correlation: {correlation:.2f}")
    
            return correlation
    
        except Exception as e:
            print(f"An error occurred in revenue_price_correlation: {e}")
            return None

    def revenue_brand_correlation(self):
        """
        Calculates and visualizes revenue by brand and optionally calculates correlation between revenue and units sold.
        """
        try:
            revenue = (
                self.cleaned_data.groupby('Brand')['Sellingprice']
                .sum()
                .reset_index()
                .rename(columns={'Sellingprice': 'Revenue'})
            )
            
            # Calculate number of units sold by brand
            units_sold = (
                self.cleaned_data.groupby('Brand').size()
                .reset_index()
                .rename(columns={0: 'UnitsSold'})
            )
            
            # Merge revenue and units sold
            merged_df = pd.merge(revenue, units_sold, on='Brand', how='left')
            
            correlation = merged_df['Revenue'].corr(merged_df['UnitsSold'])
            print(f"Correlation between Revenue and Units Sold: {correlation:.2f}")
            
            # Visualization: Revenue by Brand
            fig = px.bar(
                merged_df, 
                x='Brand', 
                y='Revenue', 
                title='Revenue by Car Manufacturer',
                labels={'Revenue': 'Revenue', 'Brand': 'Brand'}
            )
            fig.update_layout(width=800, height=600)
            fig.show()
            
            # Visualization: Correlation between Units Sold and Revenue
            fig = px.scatter(
                merged_df, 
                x='UnitsSold', 
                y='Revenue', 
                trendline='ols', 
                title=f'Correlation between Units Sold and Revenue: {correlation:.2f}',
                labels={'UnitsSold': 'Units Sold', 'Revenue': 'Revenue'}
            )
            fig.update_layout(width=800, height=600)
            fig.show()
            
            # Output the merged DataFrame
            print("Revenue and Units Sold by Brand:")
            print(merged_df)

            return merged_df

        except Exception as e:
            print(f"An error occurred in revenue_brand_correlation: {e}")
            return None

    def revenue_condition_correlation(self):
        revenue = (
            self.cleaned_data.groupby('Condition')['Sellingprice']
            .sum()
            .reset_index()
            .rename(columns={'Sellingprice': 'Revenue'})
        )
        
        # Step 2: Calculate the correlation between 'Condition' and 'Revenue'
        correlation = revenue['Condition'].corr(revenue['Revenue'])
        
        # Step 3: Visualization
        fig = px.scatter(
            revenue, 
            x='Condition', 
            y='Revenue', 
            trendline='ols', 
            title=f'Correlation between Car Condition and Revenue: {correlation:.2f}',
            labels={'Condition': 'Car Condition', 'Revenue': 'Revenue'}
        )
        fig.update_layout(width=800, height=600)
        fig.show()

        return correlation

    
    def price_odometer_correlation(self):
        correlation = self.cleaned_data['Odometer'].corr(self.cleaned_data['Sellingprice'])
        
        # Step 2: Visualization
        fig = px.scatter(
            self.cleaned_data, 
            x='Odometer', 
            y='Sellingprice', 
            trendline='ols', 
            title=f'Correlation between Odometer and Selling Price: {correlation:.2f}',
            labels={'Odometer': 'Odometer Reading (miles)', 'Sellingprice': 'Selling Price'}
        )
        fig.update_layout(width=800, height=600)
        fig.show()

        return correlation



# Instantiating the classes and loading the dataset

In [ ]:
if __name__ == "__main__":
    raw_file_path = '../dataset/raw/car_prices.csv'
    new_file_path = '../dataset/cleaned/cleaned_car_sales_data.csv'
    
    data_loader = LoadDataset(raw_file_path)
    
    data_content = data_loader.read_file()

        
    if data_content is not None:
        data_cleaner = DataCleaner(data_content)
        print(data_content.columns)
    

# Getting Unique Colour Names Before Normalisation

In [ ]:
        print(data_content['color'].unique())

In [ ]:
        print(data_content['make'].unique())

In [ ]:
        print(data_content['transmission'].unique())

In [ ]:
        print(data_content['condition'].unique())

In [ ]:
        print(data_content['interior'].unique())

# Getting the total number of rows

In [ ]:
        number_of_rows = data_content.shape[0]
        print(number_of_rows)

# Getting the total number of columns

In [ ]:
        number_of_columns = data_content.shape[1]
        print(number_of_columns)

# Getting columns' data types

In [ ]:
        print(data_content.dtypes) 

# Some information about the dataframe

In [ ]:
        print(data_content.info())    

# describe() generates descriptive statistics

In [ ]:
        print(data_content.describe())

# 15% of rows contain at least one null value

In [ ]:
        # Count the number of null rows
        num_null_rows = data_content.isnull().any(axis=1).sum()
        # the formula
        percent_of_total_rows = int((num_null_rows/number_of_rows) * 100)
        
        print(f'Number of rows with at least one null value: {num_null_rows}/{number_of_rows}\n{percent_of_total_rows}% of rows contain null values')

# There is no row with all null values

In [ ]:
        # Count the number of null rows
        num_all_null_rows = data_content.isnull().all(axis=1).sum()
        # the formula
        percent_of_total_rows = int((num_all_null_rows/number_of_rows) * 100)
        
        print(f'Number of rows with all values as null: {num_all_null_rows}/{number_of_rows}\n{percent_of_total_rows}% of rows contain all null values')

# There is no column with all null values

In [ ]:
        num_all_null_columns = data_content.isnull().all(axis=1).sum()
        percent_of_total_cols = int((num_all_null_columns/number_of_columns) * 100)
        
        print(f'Number of columns with all values as null: {num_all_null_columns}/{number_of_columns}\n{percent_of_total_cols}% of rows contain all null values')

# There are no duplicate rows

In [ ]:
        cleaner = data_cleaner.remove_duplicates()

# Renaming the header titles

In [ ]:
        cleaner = data_cleaner.rename_headers()

# Capitalising the header titles

In [ ]:
        cleaner = data_cleaner.capitalise_headers()
        print(f"New header titles: {cleaner.columns}")

In [ ]:
        cleaner = data_cleaner.normalise_brands()

In [ ]:
        cleaner = data_cleaner.normalise_colors()

In [ ]:
        cleaner = data_cleaner.normalise_transmissions()

In [ ]:
        cleaner = data_cleaner.normalise_states()

# Validating the data type of the columns

In [ ]:
        cleaner = data_cleaner.validate_values()
        print(cleaner)

# Saving the changes after cleaning the data into a new file

In [ ]:
        cleaned_data = data_cleaner.save_changes()

# Instantiating the explorer class


In [ ]:
        explorer = DataExplorer(cleaned_data)

# Total Yearly Revenue over Time
- There was a sharp increase between 2010 and 2011
- As a follow-up, 2011 to 2012 witnessed the most massive increase in sales
- 2002 was the year where the bullish trend started 
- 2014 saw the sharpest decrease in sales until 2015, going from 1,628,672M to  244,721K

In [ ]:
        annual_sales = explorer.sales_by_year()
        print(f"The sales by year: {annual_sales}")

# Top 10 Selling Car Makers in the United States from 1982 to 2015
- BMW 
- Mercedes-Benz
- Audi
- Ford
- Chevrolet
- Volkswagen
- Toyota
- Nissan
- Porsche
- Land Rover

In [ ]:
        top_10_selling_brands = explorer.top_10_selling_brands()
        print(f"Sales by Brand: {top_10_selling_brands}")

# Top 10 Most Expensive Cars
- Ferrari 458 Italia Base 2011
- Rolls-Royce Ghost EWB 2012
- BMW i8 Base 2014
- Bentley Continental GTC V8 2013
- Mercedes-Benz SLS AMG GT Base 2013
- Ferrari California Base 2012
- Bentley Flying Spur Base 2014
- Rolls-Royce Ghost Base 2010
- Mercedes-Benz S-Class S63 AMG 4MATIC 2014
- Bentley Continental GTC Base 2013

In [ ]:
        top_10_most_expensive_cars = explorer.top_10_most_expensive_cars()
        print(f"Most Expensive Cars: {top_10_most_expensive_cars}")

In [ ]:
        top_10_most_profitable_cars = explorer.top_10_most_profitable_cars()
        print(f"Top 10 Most Profitable Cars: ${top_10_most_profitable_cars}")

# Top 10 Least Profitable Cars
- Volkswagen Jetta GT 1998
- Saab 9-3 Vector 2003 automatic
- Suzuki Esteem GL 1999 automatic
- Acura Legend L 1993 automatic
- Dodge Intrepid SXT 2004 automatic
- Ford Contour GL 1996 automatic
- BMW 3 Series 328is 1996 manual
- Toyota Camry Deluxe 1991 automatic
- Toyota Paseo Base 1992 automatic
- Ford explorer awd xlt 1997 automatic 

In [ ]:
        top_10_least_profitable_cars = explorer.top_10_least_profitable_cars()
        print(f"Top 10 Least Profitable Cars: ${top_10_least_profitable_cars}")

# Which One is More Popular? Manual or Automatic Transmission? 
- Automatic accounts for 83.4% of total revenue against 6.57% for manual vehicles, and 10% others

In [ ]:
        sales_by_transmission = explorer.sales_by_transmission()
        print(f"Sales by Transmission: {sales_by_transmission}")

# Total Sales 
Total Sales: $57.00M

In [ ]:
        total_sales = explorer.total_sales()
        print(f"Total Sales: ${total_sales}")

# Top 10 Sellers by Revenue
- Financial Services Remarketing (lease)
- High Bid Crading CO inc
- JPMorgan Chase Bank N.A.
- TDAF Remarketing
- R Hollenshead Auto Sales inc
- Nissan North America inc.
- The Hertz Corporation
- Mercedes-Benz USA
- Fiserv/USB Dealer Services Northstar Exchange

In [ ]:
        top_10_sellers_by_revenue = explorer.top_10_sellers_by_revenue()
        print(f"Top 10 Sellers: ${top_10_sellers_by_revenue}")

# Top 10 Sellers with the Lowest Generated Revenue
- Lafontaine Cadillac buick GMC inc
- Mossy Nissan Oceanside
- Nissan of New Orleans
- Team Toyota
- Deemiracle Auto Group llc
- Healey Chrysler Dodge Jeep llc
- Simer Used Car Sales
- Mid City Motors
- Lithia Chrysler Jeep Dodge
- Poway Honda

In [ ]:
        top_10_worst_sellers_by_revenue = explorer.top_10_worst_sellers_by_revenue()
        print(f"Top 10 Worst Sellers: ${top_10_worst_sellers_by_revenue}")

# Top 10 States with the Highest Generated Revenue
- California (the most profitable state by a landslide)
- Florida
- Pennsylvania
- Texas
- Georgia
- Ohio
- Illinois
- New Jersey
- Tennessee
- Nevada

In [ ]:
        revenue_by_state = explorer.revenue_by_state()
        print(f"Revenue by State: ${revenue_by_state}")

# States with the Highest Prices on Average
- Mississippi (Despite having the highest prices on average and being an affordable state, it's not amongst the top 10 most profitable states. This could indicate the presence of outliers in the Mississippi data or irrationally high prices.)
- Pennsylvania
- Tennessee
- Wisconsin
- Ohio
- California
- Colorado
- Texas
- Florida
- Georgia

In [ ]:
        average_prices_by_state = explorer.average_prices_by_state()
        print(f"Average Prices by State: ${average_prices_by_state}")

# Top 10 Most Popular Colours
(oddly named colours were replaced with "unknown" due to possible mistakes in data entry)
- Black
- White
- Silver
- Grey
- Blue
- Red
- Green
- *unknown* 
- Gold
- Beige

In [ ]:
        most_popular_colors = explorer.most_popular_colors()
        print(f"{most_popular_colors}")

In [ ]:
        correlation = DataCorrelation(cleaned_data)

# Did the Average Selling Prices Affect the Total Revenue?

- <b> 0 to 0.3: Weak positive correlation.</b>
- 0.3 to 0.7: Moderate positive correlation.
- 0.7 to 1: Strong positive correlation.
- 0 to -0.3: Weak negative correlation.
- 0.3 to -0.7: Moderate negative correlation.
- 0.7 to -1: Strong negative correlation.



<b>Since there is a weak yet positive correlation between the prices of the vehicles, thus, selling prices don't seem to be
a strong influencer on the revenue in different states. We make the conclusion that the prices have been increasing rationally throughout years</b>

In [ ]:
        rev_price_corr = correlation.revenue_price_correlation()
        print(f"Correlation: {rev_price_corr}")

# The Correlation between Number of Cars sold and Total Revenue (0.83 -> Strong positive correlation.)
- The more car units brands sell, the higher revenue they generate. So there is a postive and direct relationship between the number of vehicles they sell and their total revenue.

In [ ]:
        brand_revenue_df = correlation.revenue_brand_correlation()
        print("Revenue and Units Sold by Brand:")
        print(brand_revenue_df)

# The Correlation between Car Conditions and Revenue (0.5 -> Moderate positive correlation.)
- It could be said that the condition of the car has a fairly direct relationship with the revenue. So the customers most likely will
  not overlook the condition of the car during the purchase. The better the condition of the car, the higher chance of getting sold.

In [ ]:
        rev_condition_corr = correlation.revenue_condition_correlation()
        print(f"Correlation: {rev_condition_corr}")

# The Correlation between Odometer and the Selling Price of vehicles (-0.5 -> Moderate negative correlation.)
- The distance (odometer) a car travelled does not seem to be an massive influencer on the selling price of vehicles. The customers
    seem to ignore the figure displayed by odometer

In [ ]:
        price_odometer_corr = correlation.price_odometer_correlation()
        print(f"Correlation: {price_odometer_corr}")